# 01. 데이터 수집

실손보험 관련 온라인 질문 1,000건 수집 코드

공모전 당시 사용한 수집 로직을 공개용으로 정리한 버전이며, 원문 데이터는 저장소에 포함하지 않음  
사이트 API 구조가 변경된 경우 일부 수정이 필요할 수 있음

In [ ]:
from pathlib import Path
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://post.api.a-ha.io/questions"
START_UUID = "45325f558979831a9ce84adedbc1784e"
TARGET = 1000

OUTPUT_PATH = Path("data/aha_medical_insurance_1000.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
}

params = {
    "uuid": START_UUID,
    "answeredOnly": "true",
    "releasedOnly": "true",
    "topic": "94",
}

In [2]:
questions = []
seen_uuids = set()
page = 1

while len(questions) < TARGET:
    response = requests.get(
        BASE_URL,
        params=params,
        headers=headers,
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()

    if not data:
        break

    new_count = 0

    for item in data:
        uuid = item.get("uuid")

        if not uuid or uuid in seen_uuids:
            continue

        seen_uuids.add(uuid)

        body_html = item.get("body") or ""
        body = BeautifulSoup(
            body_html,
            "html.parser",
        ).get_text(" ", strip=True)

        questions.append({
            "uuid": uuid,
            "title": (item.get("title") or "").strip(),
            "body": body,
            "createdAt": item.get("createdAt"),
            "categoryId": item.get("categoryId"),
        })

        new_count += 1

        if len(questions) >= TARGET:
            break

    if page % 10 == 0 or len(questions) >= TARGET:
        print(f"{len(questions):,} / {TARGET:,}")

    if new_count == 0:
        break

    next_uuid = data[-1].get("uuid")

    if not next_uuid or next_uuid == params["uuid"]:
        break

    params["uuid"] = next_uuid
    page += 1
    time.sleep(1.5)

100 / 1,000
200 / 1,000
300 / 1,000
400 / 1,000
500 / 1,000
600 / 1,000
700 / 1,000
800 / 1,000
900 / 1,000
1,000 / 1,000


In [3]:
df = pd.DataFrame(questions)

df["question"] = (
    df["title"].fillna("")
    + " "
    + df["body"].fillna("")
).str.strip()

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("수집 건수:", len(df))
print("고유 UUID:", df["uuid"].nunique())
print("저장:", OUTPUT_PATH)

수집 건수: 1000
고유 UUID: 1000
저장: data\aha_medical_insurance_1000.csv


**당시 실행 결과**

- 수집 질문 1,000건
- 고유 UUID 1,000건

분석에 사용된 원문은 사용자 작성 콘텐츠를 포함하므로 공개 저장소에는 업로드하지 않음